# 06 · 把 `cowork_worker` 迁到 Copilot SDK —— 设计与最小原型

本 notebook 把现有 `ref/hosted_agent_foundry/cowork_worker` 的各个组件，逐一映射到 Copilot Python SDK 的对应能力，并给出一个最小可运行原型与迁移风险清单。

## 0. 一图看懂迁移：Foundry 栈 → Copilot SDK 栈

```mermaid
flowchart TB
    subgraph Before["现在：Foundry + agent_framework"]
        BE1["ResponsesHostServer<br/>(FastAPI)"]
        BE2["Agent (agent_framework)"]
        BE3["CoworkFoundryChatClient<br/>(Azure AAD)"]
        BE4["SkillsContextProvider<br/>(注入 &lt;available_skills&gt;)"]
        BE5["WorkspaceMiddleware<br/>(解析 tag · chdir · enabled_skills)"]
        BE6["make_tools()<br/>bash / read_file / write_file /<br/>edit_file / save_artifact / python_tool ..."]
        BE7["PER_TURN_MODEL contextvar"]
        BE1 --> BE2 --> BE3
        BE2 --> BE4
        BE2 --> BE5
        BE2 --> BE6
        BE3 --> BE7
    end

    subgraph After["迁完：Copilot SDK"]
        AF1["FastAPI<br/>+ async generator → SSE"]
        AF2["CopilotClient<br/>(整个 worker 1 个)"]
        AF3["SessionPool<br/>key=(ws_id, model)"]
        AF4["hooks:<br/>on_session_start<br/>on_user_prompt_submitted<br/>on_pre_tool_use"]
        AF5["内建工具:<br/>bash · view · apply_patch · rg ..."]
        AF6["@define_tool<br/>(仅保留 python_tool 沙箱<br/>+ save_artifact)"]
        AF7["provider= Azure OpenAI (key)"]
        AF1 --> AF2 --> AF3 --> AF4
        AF3 --> AF5
        AF3 --> AF6
        AF3 --> AF7
    end

    Before -. "1500 行 → 300-400 行" .-> After
```

下面 §1 重新梳理你现在系统里"关键胶水"代码，§2 给出每块对应的 SDK 落点，§3 画目标架构，§4 跑最小原型，§5/6 是迁移阶段建议 + 风险清单。


## 1. 现状回顾

```
ResponsesHostServer (FastAPI/agent_framework_foundry_hosting)
  └─ Agent (agent_framework)
       ├─ chat_client = CoworkFoundryChatClient (FoundryChatClient 子类)
       ├─ context_providers = [SkillsContextProvider]
       ├─ middleware = [WorkspaceMiddleware]   ← AgentMiddleware（必须）
       └─ tools = make_tools()  → bash / python_tool / read_file / write_file /
                                  save_artifact / load_artifact / run_skill / ...
```

**关键“胶水”**：
1. 用户消息里夹带 inline tag：`[workspace_id=...] [session_id=...] [model=...]`
2. `WorkspaceMiddleware` 解析 tag → 设 contextvars + env vars → `os.chdir(workspace_dir)` → 从 Cosmos 读 `enabled_skills` 注入 `SkillsContextProvider`
3. `SkillsContextProvider.before_run` 把 `<available_skills>` / `<available_files>` 系统块挂到 `SessionContext.context_messages`
4. `CoworkFoundryChatClient` 在每个 chat round 读 `PER_TURN_MODEL` 让模型可按 turn 切换
5. `_strip_cowork_tags` 在用模型前把 tag 从消息里抹掉（避免污染 history/trace）

## 2. 模块对照表（Foundry → Copilot SDK）

| 现 cowork_worker | Copilot SDK 对应 | 备注 |
|---|---|---|
| `FoundryChatClient` + Azure AAD | `CopilotClient(SubprocessConfig(...))` + `provider={'type':'azure', ...}` | **BYOK 只支持 key**，要从 MSI 切回 API key |
| `Agent(..., tools=...)` | `client.create_session(tools=[...], system_message=..., model=...)` | 每个 (workspace, model) 起独立 session |
| `AgentMiddleware.process` | `hooks.on_user_prompt_submitted` (改 prompt / 抹 tag) + `hooks.on_session_start` (注入 skills) | 用 hook 替代 middleware |
| `SkillsContextProvider.before_run` 输出 `<available_skills>` | `on_user_prompt_submitted` 返回 `additionalContext` 或 `modifiedPrompt` 拼接 | 不再有 ContextProvider 概念 |
| `<available_files>` 系统块 | 同上；或开启 `infinite_sessions` 让 SDK 管 workspace + `view` 工具读盘 | SDK 自带文件 IO 工具 |
| `CoworkFoundryChatClient._check_model_presence` + `PER_TURN_MODEL` | 每个 workspace/model 组合一个 session（key=(wid, model)）| 天然并发安全 |
| `_strip_cowork_tags` | `on_user_prompt_submitted` 里用同样的 regex 改 `modifiedPrompt`；外面 tag 解析照旧 | tag 只是传输层，hook 同样能截到 |
| `WORKSPACE_ID/SESSION_ID` contextvar + env | 推荐通过 hook 在执行 tool 前 chdir + 设 env；或在 `on_pre_tool_use` 里改 `modifiedArgs` | hook 总能拿到 sessionId |
| `make_tools()` 自定义工具 | `@define_tool` + Pydantic 重写 | bash/python_tool 可保留自定义（带资源限额），或直接用 SDK built-in `bash` + `view`/`edit_file` |
| `ResponsesHostServer` (Foundry Responses API) | 自写 FastAPI 路由：把 `session.send + Queue` 流出为 SSE | 见 notebook 02 的 async generator |
| `stage_workspace_manifest` + 手工 workspace 目录 | `infinite_sessions={'enabled': True}` + `session.workspace_path` | SDK 已经给每个 session 分配 `~/.copilot/session-state/<sid>/` |
| `enable_instrumentation`（App Insights） | `SubprocessConfig(telemetry={'otlp_endpoint': ...})` | 标准 OTLP，可指 App Insights 的 ADX endpoint 或本地 collector |
| 权限 / 工具白名单 | `on_permission_request` + `available_tools` / `excluded_tools` | 比现在精细 |

## 3. 目标架构

```
FastAPI app (uvicorn)
  ├─ startup: CopilotClient(SubprocessConfig(telemetry=..., env=...))
  │     await client.start()
  ├─ POST /chat
  │     1. 从 body 取 workspace_id / session_id / model / prompt
  │     2. session = await session_pool.get_or_create(workspace_id, model)
  │     3. async for chunk in stream_text(session, prompt): yield SSE
  └─ shutdown: await client.stop()

SessionPool
  key = (workspace_id, model)
  value = CopilotSession (已带好 hooks + tools + provider)

Hooks 装到每个 session 上：
  on_session_start         → chdir workspace_path/<wid>，读 enabled_skills，stash 到 closure
  on_user_prompt_submitted → 抹 cowork tag，拼 <available_skills><available_files> 进 prompt
  on_pre_tool_use          → 按需 deny（替代 _LLM_ENV_WHITELIST 的安全网）
```

## 4. 最小原型：单 workspace、单 model、自定义两个工具

下面这个 cell **不连 Foundry**，纯 SDK 跑通，用来验证迁移路径可行。

迁完后一次 `POST /chat` 在新栈里的时序：

```mermaid
sequenceDiagram
    autonumber
    participant FE as Frontend
    participant API as FastAPI /chat
    participant Pool as SessionPool
    participant Sess as CopilotSession
    participant CLI as copilot CLI
    participant LLM as Azure OpenAI

    FE->>API: POST /chat<br/>{ws_id, model, prompt}
    API->>API: 解析 [workspace_id=...] tag
    API->>Pool: get_or_create(ws_id, model)
    Pool-->>Sess: 复用 / 新建 session
    Note over Sess,CLI: 首次创建会触发 on_session_start<br/>(chdir + 注入 skills)
    API->>Sess: stream_text(session, prompt)
    Sess->>CLI: send (经 on_user_prompt_submitted 改写)
    CLI->>LLM: chat.completions
    LLM-->>CLI: delta + tool_call
    CLI-->>API: events → Queue → async for
    API-->>FE: SSE chunks
    CLI-->>API: session.idle
    API-->>FE: SSE [DONE]
```


In [ ]:
import os, re, asyncio, json
from pathlib import Path
from typing import Any
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from copilot import CopilotClient, define_tool
from copilot.session import PermissionHandler
from copilot.generated.session_events import AssistantMessageData, SessionIdleData

# 加载 .env 获取 GPT 5.4 端点
load_dotenv()

TAG_RE = re.compile(r'^\s*(?:\[(?:workspace_id|session_id|model)=[\w\-./]+\]\s*)+')

# --- 1) 自定义工具：save_artifact / list_artifacts ---------------------------
class SaveArtifactParams(BaseModel):
    name: str = Field(description='Artifact filename, relative path allowed')
    content: str = Field(description='Text content to write')

@define_tool(description='Save text artifact under the current workspace output/ dir', skip_permission=False)
async def save_artifact(params: SaveArtifactParams) -> str:
    out = Path(os.environ.get('WORKSPACE_DIR', '.')) / 'output' / params.name
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(params.content)
    return f'saved {out} ({len(params.content)} chars)'

class ListArtifactsParams(BaseModel):
    pass

@define_tool(description='List artifacts in the current workspace output/ dir', skip_permission=True)
async def list_artifacts(_: ListArtifactsParams) -> str:
    out = Path(os.environ.get('WORKSPACE_DIR', '.')) / 'output'
    if not out.exists():
        return '(no artifacts)'
    return '\n'.join(sorted(p.name for p in out.iterdir()))

# --- 2) Hook：抹 tag + 注入 <available_skills> -------------------------------
ENABLED_SKILLS = ['summarize', 'translate', 'extract_kv']  # 模拟 Cosmos 读出来

async def on_user_prompt_submitted(input_: dict, invocation: Any) -> dict:
    prompt = input_['prompt']
    # 抹 tag
    cleaned = TAG_RE.sub('', prompt)
    # 拼系统块（也可以走 additionalContext）
    skills_block = '<available_skills>\n' + '\n'.join(f'- {s}' for s in ENABLED_SKILLS) + '\n</available_skills>'
    return {'modifiedPrompt': f'{skills_block}\n\n{cleaned}'}

async def on_session_start(input_: dict, invocation: Any) -> dict:
    wid = os.environ.get('WORKSPACE_ID', 'demo-ws')
    wsdir = Path('/tmp') / 'cowork-sdk' / wid
    wsdir.mkdir(parents=True, exist_ok=True)
    (wsdir / 'output').mkdir(exist_ok=True)
    os.environ['WORKSPACE_DIR'] = str(wsdir)
    os.chdir(wsdir)
    return {'additionalContext': f'workspace ready at {wsdir}'}

In [ ]:
# --- 3) 跑起来 ---------------------------------------------------------------
async def run_one_turn(prompt: str):
    provider = {
        'type': 'azure',
        'base_url': os.environ['AZURE_OPENAI_ENDPOINT_GPT_5_4'],
        'api_key': os.environ['AZURE_OPENAI_API_KEY_GPT_5_4'],
        'azure': {'api_version': os.getenv('AZURE_OPENAI_API_VERSION_GPT_5_4', '2025-03-01-preview')},
    }
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=provider,
            tools=[save_artifact, list_artifacts],
            hooks={
                'on_session_start': on_session_start,
                'on_user_prompt_submitted': on_user_prompt_submitted,
            },
            system_message={
                'mode': 'append',
                'content': 'You are the cowork agent. Use the provided skills and save outputs via save_artifact.',
            },
        ) as session:
            done = asyncio.Event()
            def on_event(e):
                match e.data:
                    case AssistantMessageData() as d:
                        print('\nASSISTANT:', d.content)
                    case SessionIdleData():
                        done.set()
            session.on(on_event)
            await session.send(prompt)
            await done.wait()

os.environ['WORKSPACE_ID'] = 'ws-demo-001'
await run_one_turn('[workspace_id=ws-demo-001] [model=gpt-5.4] Save a haiku about octopus engineers to haiku.txt, then list artifacts.')

## 5. 迁移阶段建议

1. **POC**：跑通本 notebook 中的最小原型（BYOK→Azure，自定义工具，hook 解析 tag）
2. **平行实现**：在 `ref/hosted_agent_foundry_ghcp_sdk/` 下建 `cowork_worker_sdk/` 目录，复刻 `worker/` 子模块：
   - `runtime.py` → `CopilotClient` 单例 + `SessionPool`
   - `hooks.py` → `on_session_start` / `on_user_prompt_submitted` / `on_pre_tool_use`
   - `tools/*.py` → 把 `worker/tools/*.py` 用 `@define_tool` 重新声明（核心逻辑不动）
   - `server.py` → FastAPI，提供 `/chat`、`/sessions/{id}` 两个端点；用 async generator → SSE
3. **行为对齐**：用 `agile-agents-eval/` 现有评测集跑 A/B（原 Foundry vs SDK）
4. **切流量**：在 backend 网关层加 feature flag，按 workspace 灰度
5. **下线**：删除 `FoundryChatClient` / `ResponsesHostServer` 依赖，缩 Dockerfile

## 6. 已知风险 / 待确认

| 风险 | 说明 | 缓解 |
|---|---|---|
| BYOK 不支持 Managed Identity | 现在生产用 `DefaultAzureCredential` 走 AAD；SDK 强制 key | 在 Key Vault 拉 key，启动时注入 |
| 同一 session 不能换 model | `PER_TURN_MODEL` 那套行为变成“每个 model 一个 session” | SessionPool 加 `(wid, model)` key；新 model 触发新 session |
| `_strip_cowork_tags` 还得改 round 2+ 的 system context | SDK hook 只能改“用户当前 prompt”；后续 round 不再触发 `on_user_prompt_submitted` | 在 backend 永远只在最新 user message 加 tag；hook 抹一次即可 |
| Foundry 的 hosted tools（web search / code interpreter） | SDK 没有完全对等的 web search；code interpreter 可用内置 `bash`+`python` | 接 Bing Search MCP，或保留一个走 Foundry 的“工具桥”declaration-only tool |
| Public Preview API 可能变 | SDK 本身处于 preview | pin 版本，CI 跑兼容测试 |
| 子进程 + bundled CLI 的资源开销 | 一个 CopilotClient = 一个 CLI 子进程 | 每个 worker 一个 client；session 由 pool 复用，不要为每个请求重启 client |
| 现有 `python_tool` 走自定义 venv + rlimit | SDK 内置 `bash` 不带 rlimit 沙箱 | 保留自定义 `python_tool` 工具（`overrides_built_in_tool=False`，新工具名），在 `excluded_tools` 里关 SDK 的 `bash` |

## 7. 下一步

- [ ] 在本地 `.env` 准备 Azure OpenAI key，跑通本 notebook 第 4 节最小原型
- [ ] 评估 SDK 自带 `view` / `edit_file` / `bash` 工具 vs 现有自研工具的覆盖度
- [ ] 决定权限策略：现在 `approve_all`；生产建议按 `kind` 白名单（`shell` 拒、`write` 限 `output/` 子树）
- [ ] 用 `agile-agents-eval` 现有测试集做 A/B